In [3]:
import os
import numpy as np
import nibabel as nib
from tqdm import tqdm

# ================= CONFIG =================
dataset_root = "/home/23bq1a4293/Personal/project/seg_data/meningioma/BraTS-MEN-Train"
patients = sorted([d for d in os.listdir(dataset_root) if os.path.isdir(os.path.join(dataset_root, d))])

print(f"Found {len(patients)} total patients in dataset root.")
print("Analyzing 300 patients to compute dataset statistics...\n")

stats = {
    "total_patients": 0,
    "has_nec": 0,
    "has_ede": 0,
    "has_enh": 0,
    "vol_nec": [],
    "vol_ede": [],
    "vol_enh": [],
    "bbox_w": [],
    "bbox_h": [],
    "bbox_d": []
}

# Analyze a subset (first 300) to save time, it's enough for statistical significance
for pid in tqdm(patients[:300], desc="Analyzing Dataset"):
    label_path = os.path.join(dataset_root, pid, f"{pid}-seg.nii.gz")
    if not os.path.exists(label_path):
        continue
        
      # Load label (Corrected)
    label = nib.load(label_path).get_fdata(dtype=np.float32).astype(np.uint8)
    stats["total_patients"] += 1
    
    # Calculate Volumes (Voxel counts)
    v_nec = np.sum(label == 1)
    v_ede = np.sum(label == 2)
    v_enh = np.sum(label == 3)
    
    if v_nec > 0:
        stats["has_nec"] += 1
        stats["vol_nec"].append(v_nec)
    if v_ede > 0:
        stats["has_ede"] += 1
        stats["vol_ede"].append(v_ede)
    if v_enh > 0:
        stats["has_enh"] += 1
        stats["vol_enh"].append(v_enh)
        
    # Calculate Bounding Box of the entire tumor (Classes 1, 2, 3 combined)
    tumor_coords = np.argwhere(label > 0)
    if len(tumor_coords) > 0:
        # Nifti shapes are typically (W, H, D)
        w_min, h_min, d_min = tumor_coords.min(axis=0)
        w_max, h_max, d_max = tumor_coords.max(axis=0)
        
        stats["bbox_w"].append(w_max - w_min)
        stats["bbox_h"].append(h_max - h_min)
        stats["bbox_d"].append(d_max - d_min)

# ================= PRINT REPORT =================
print("\n" + "="*60)
print(f"🧠 MENINGIOMA DATASET STATISTICS ({stats['total_patients']} Patients)")
print("="*60)

# Class Presence
print("\n--- Class Presence (How many patients have it?) ---")
print(f"Necrotic Core (Class 1):     {stats['has_nec']} / {stats['total_patients']} ({(stats['has_nec']/stats['total_patients'])*100:.1f}%)")
print(f"Peritumoral Edema (Class 2): {stats['has_ede']} / {stats['total_patients']} ({(stats['has_ede']/stats['total_patients'])*100:.1f}%)")
print(f"Enhancing Tumor (Class 3):   {stats['has_enh']} / {stats['total_patients']} ({(stats['has_enh']/stats['total_patients'])*100:.1f}%)")

# Average Volumes
print("\n--- Average Size of Tumor Components (when present) ---")
print(f"Necrotic Core:     {np.mean(stats['vol_nec']):.0f} voxels")
print(f"Peritumoral Edema: {np.mean(stats['vol_ede']):.0f} voxels")
print(f"Enhancing Tumor:   {np.mean(stats['vol_enh']):.0f} voxels")

# Bounding Boxes
print("\n--- Bounding Box Analytics (For Patch Size Selection) ---")
print(f"Median Tumor Size (W x H x D): {np.median(stats['bbox_w']):.0f} x {np.median(stats['bbox_h']):.0f} x {np.median(stats['bbox_d']):.0f}")
print(f"Max Tumor Size    (W x H x D): {np.max(stats['bbox_w'])} x {np.max(stats['bbox_h'])} x {np.max(stats['bbox_d'])}")
print(f"90th Percentile   (W x H x D): {np.percentile(stats['bbox_w'], 90):.0f} x {np.percentile(stats['bbox_h'], 90):.0f} x {np.percentile(stats['bbox_d'], 90):.0f}")
print("="*60)

Found 1000 total patients in dataset root.
Analyzing 300 patients to compute dataset statistics...



Analyzing Dataset:   0%|          | 0/300 [00:00<?, ?it/s]

Analyzing Dataset: 100%|██████████| 300/300 [01:10<00:00,  4.25it/s]


🧠 MENINGIOMA DATASET STATISTICS (300 Patients)

--- Class Presence (How many patients have it?) ---
Necrotic Core (Class 1):     108 / 300 (36.0%)
Peritumoral Edema (Class 2): 158 / 300 (52.7%)
Enhancing Tumor (Class 3):   299 / 300 (99.7%)

--- Average Size of Tumor Components (when present) ---
Necrotic Core:     4409 voxels
Peritumoral Edema: 41016 voxels
Enhancing Tumor:   19589 voxels

--- Bounding Box Analytics (For Patch Size Selection) ---
Median Tumor Size (W x H x D): 33 x 40 x 34
Max Tumor Size    (W x H x D): 128 x 163 x 114
90th Percentile   (W x H x D): 71 x 104 x 81
